<cell_type>markdown</cell_type># 01a — Write Delta natively (path-reference)

**Purpose:** Generate the synthetic multimodal dataset and land it the Databricks-native way: **JPEG files in a Volume** plus a **governed Delta table** that references them by path (`image_path`) alongside the metadata columns. This is one of the two **paved-path native-ingest routes** — `01a` (this notebook) writes the source to Delta path-refs; `01b_lance_native.ipynb` writes the *same* source straight to Lance. Both begin from the same deterministic generation and diverge only at the terminal format, so `01a` vs `01b` is a clean one-source → two-formats comparison.

Covers **generation + the Delta write + the Delta-side ETL backfill**. The head-to-head comparison lives in `03_compile_results.ipynb`.

| | Delta (this notebook) |
|---|---|
| Image storage | JPEG files in a Volume, referenced by **`image_path`** |
| Writer | Ray writes files + `write_databricks_table` (via SQL Warehouse) |
| Add a column | `ALTER TABLE ADD COLUMN` + full backfill |

**One widget, `delta_write_mode`, picks what to write:**

- **`pathref`** (default) — the paved path: JPEG files in a Volume + a path-reference Delta table (`synthetic_delta_{size}`, the dataset `02` trains on).
- **`inline`** — the anti-pattern: JPEG bytes stored **inline** in a `binary` column, in a separate `synthetic_delta_inline_{size}` table (see the section at the end).
- **`both`** — writes both from one generation pass.

**When `both`**, generation is materialized once and teed to both writes (no regeneration). **When a single mode**, the dataset is kept *lazy* and streams straight into that write — nothing is parked in the object store, so a single-path run at 1m+ doesn't spill just to hold a copy it won't reuse. (Trade-off: a lazy single-path write timer includes generation, which is identical work for both formats and not the bottleneck — the file PUTs / Spark funnel are.)

**Compute:** Databricks Classic Compute — 8 worker nodes × 16 CPUs; see the Ray-cluster cell for the Ray/Spark split.

---

**Outputs (per size tier):**
- *(`pathref` / `both`)* JPEG files at `/Volumes/{catalog}/{schema}/{volume}/synthetic_images_{size}/`; Delta table `{catalog}.{schema}.synthetic_delta_{size}` (metadata + `image_path` + `embedding_norm`) — the paved Delta dataset `02_training_benchmark.ipynb` trains on; metrics JSON `artifacts/delta_{size}.json`
- *(`inline` / `both`)* inline table `{catalog}.{schema}.synthetic_delta_inline_{size}` + `artifacts/delta_inline_{size}.json`

**Next:** `01b_lance_native.ipynb` (the Lance-native counterpart), then `02_training_benchmark.ipynb`. If you *already* have JPEGs in a Volume + a Delta table and want to measure the cost of migrating them to Lance, see `optional/01_lance_conversion.ipynb`.

In [0]:
# Must install before setup_ray_cluster — installing after restarts the Ray workers.
# ray[data]==2.54.1 pinned: 2.55.0+ added storage_options_provider to lance_datasink,
%pip install -qU "ray[default,data]==2.54.1" pylance numpy pandas "databricks-sdk>=0.49.0"
dbutils.library.restartPython()

In [0]:
# ── Widgets ─────────────────────────────────────────────
dbutils.widgets.dropdown("size", "10k", ["10k", "100k", "1m", "10m"], "Dataset size")
dbutils.widgets.text("catalog", "main", "UC catalog")
dbutils.widgets.text("schema", "ml_benchmark", "UC schema")
dbutils.widgets.text("volume", "lance_benchmark", "UC volume")
dbutils.widgets.text("seed", "42", "RNG seed")
dbutils.widgets.text("embedding_dim", "512", "Embedding dim")
# What to write: the paved path-ref table, the inline anti-pattern table, or both.
dbutils.widgets.dropdown("delta_write_mode", "pathref", ["pathref", "inline", "both"], "Delta write mode")

size          = dbutils.widgets.get("size")
catalog       = dbutils.widgets.get("catalog")
schema        = dbutils.widgets.get("schema")
volume        = dbutils.widgets.get("volume")
SEED          = int(dbutils.widgets.get("seed"))
EMBEDDING_DIM = int(dbutils.widgets.get("embedding_dim"))
DELTA_WRITE_MODE   = dbutils.widgets.get("delta_write_mode")
WRITE_PATH_REF     = DELTA_WRITE_MODE in ("pathref", "both")
WRITE_INLINE_DELTA = DELTA_WRITE_MODE in ("inline", "both")

# Reuse the generated bytes only in "both" mode — then materialize once and tee.
# For a single path there's nothing to reuse: keep ds lazy and stream generation
# straight into the write (avoids parking ~150KB/row in the object store — which would
# spill at 1m+ and confound the write-scaling measurement).
REUSE = WRITE_PATH_REF and WRITE_INLINE_DELTA

SIZE_MAP = {"10k": 10_000, "100k": 100_000, "1m": 1_000_000, "10m": 10_000_000}
N_ROWS   = SIZE_MAP[size]

# Fixed category set — MUST match 01b_lance_native + 02_training_benchmark.
CATEGORIES = ["cat", "dog", "car", "tree", "house", "flower", "boat", "bird"]

base_vol      = f"/Volumes/{catalog}/{schema}/{volume}"
images_dir    = f"{base_vol}/synthetic_images_{size}"      # JPEG files, referenced by the Delta table
delta_table   = f"{catalog}.{schema}.synthetic_delta_{size}"
inline_delta_table = f"{catalog}.{schema}.synthetic_delta_inline_{size}"  # optional anti-pattern table
artifacts_dir = f"{base_vol}/artifacts"                    # write metrics here for notebook 03

print(f"Size tier   : {size} ({N_ROWS:,} rows)")
print(f"Write mode  : {DELTA_WRITE_MODE}")
print(f"Path-ref    : {delta_table}" + ("" if WRITE_PATH_REF else "  (skipped)"))
print(f"Inline Delta: {inline_delta_table}" + ("" if WRITE_INLINE_DELTA else "  (skipped)"))
print(f"ds strategy : {'materialize + tee (both)' if REUSE else 'lazy stream (single path)'}")
print(f"Artifacts   : {artifacts_dir}")
print(f"Categories  : {CATEGORIES}")


In [0]:
import os

# Credentials — set BEFORE setup_ray_cluster so Ray workers inherit them (Ray 2.41+).
os.environ["DATABRICKS_HOST"]  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
os.environ["DATABRICKS_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

In [0]:
# Ensure output + Ray tmp volumes exist, plus the image and artifacts dirs.
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import catalog as sdk_catalog

w = WorkspaceClient()
for vol_name in [volume, "ray_tmp"]:
    try:
        w.volumes.read(f"{catalog}.{schema}.{vol_name}")
    except Exception:
        w.volumes.create(catalog_name=catalog, schema_name=schema, name=vol_name,
                         volume_type=sdk_catalog.VolumeType.MANAGED)
        print(f"Created volume {catalog}.{schema}.{vol_name}")

ray_tmp_path = f"/Volumes/{catalog}/{schema}/ray_tmp"
os.makedirs(images_dir, exist_ok=True)
os.makedirs(artifacts_dir, exist_ok=True)
print(f"Ray tmp     : {ray_tmp_path}")
print(f"Images dir  : {images_dir}")
print(f"Artifacts   : {artifacts_dir}")


In [0]:
# Classic Compute Ray cluster.
# N_WORKER_NODES allocated to Ray; remaining nodes stay available for Spark
# (write_databricks_table, DESCRIBE DETAIL, ALTER TABLE, etc.).
import ray
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

try:
    shutdown_ray_cluster()
except Exception:
    pass

N_WORKER_NODES = 7
CPUS_PER_NODE  = 16

setup_ray_cluster(
    min_worker_nodes=N_WORKER_NODES,
    max_worker_nodes=N_WORKER_NODES,      
    num_cpus_worker_node=CPUS_PER_NODE,
    num_gpus_worker_node=0,
    collect_log_to_path=ray_tmp_path,
)
ray.init(address="auto", ignore_reinit_error=True)

total_cpus = ray.cluster_resources().get("CPU", 0)
print(f"Total CPUs  : {total_cpus:.0f} | alive nodes: {sum(1 for n in ray.nodes() if n['Alive'])}")
assert total_cpus >= N_WORKER_NODES * CPUS_PER_NODE * 0.9, "Cluster did not fully start"

In [0]:
import time, os


def dir_stats(path):
    total, nfiles = 0, 0
    for root, _, files in os.walk(path):
        for f in files:
            try:
                total += os.path.getsize(os.path.join(root, f)); nfiles += 1
            except OSError:
                pass
    return total, nfiles

## Generate synthetic data (once)

`ray.data.range(N).map_batches(generate_batch)` fans generation across the cluster. Each row is seeded by `(SEED, id)`, so generation is deterministic and independent of block partitioning. The image is conditioned on category (hue) so the classification task in `02` is learnable; noise keeps the JPEG in the ~30–300KB range.

In [0]:
import numpy as np


def _make_image(rng, category_idx, n_categories):
    """Procedural RGB image conditioned on category, JPEG-encoded to ~30-300KB."""
    import io
    from PIL import Image

    side = int(rng.integers(256, 512))
    base = np.zeros((side, side, 3), dtype=np.float32)
    hue = category_idx / n_categories
    base[..., 0] = 255 * hue
    base[..., 1] = 255 * (1 - hue)
    base[..., 2] = 128
    noise = rng.integers(0, 60, size=(side, side, 3))
    arr = np.clip(base + noise, 0, 255).astype(np.uint8)

    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, format="JPEG", quality=90)
    return buf.getvalue()


def generate_batch(batch, seed, categories, embedding_dim):
    ids = batch["id"]
    n_cat = len(categories)
    images, captions, embeddings, cats, brightness, quality = [], [], [], [], [], []
    for _id in ids:
        rng = np.random.default_rng([seed, int(_id)])
        cat_idx = int(rng.integers(0, n_cat))
        images.append(_make_image(rng, cat_idx, n_cat))
        captions.append(f"a photo of a {categories[cat_idx]} " + "x" * int(rng.integers(0, 40)))
        embeddings.append(rng.standard_normal(embedding_dim).astype(np.float32))
        cats.append(categories[cat_idx])
        brightness.append(float(rng.random()))
        quality.append(int(rng.integers(1, 6)))
    return {
        "id":         np.asarray(ids),
        "image":      np.asarray(images, dtype=object),
        "caption":    np.asarray(captions, dtype=object),
        "embedding":  np.asarray(embeddings, dtype=np.float32),
        "category":   np.asarray(cats, dtype=object),
        "brightness": np.asarray(brightness, dtype=np.float32),
        "quality":    np.asarray(quality, dtype=np.int32),
    }

In [0]:
# Build the generation pipeline. Materialize ONLY when both write paths run, so they
# share one in-memory copy; for a single path keep ds lazy so generation streams
# straight into the write with no full-dataset object-store copy.
override_blocks = max(64, N_ROWS // 5_000)

ds = ray.data.range(N_ROWS, override_num_blocks=override_blocks).map_batches(
    generate_batch,
    fn_kwargs={"seed": SEED, "categories": CATEGORIES, "embedding_dim": EMBEDDING_DIM},
    batch_size=512,
)

if REUSE:
    ds = ds.materialize()
    print(f"Generated {ds.count():,} rows (materialized — teed to both write paths)")
    # Safe: the sum pass reads the already-materialized blocks (no regeneration).
    total_image_bytes = ds.map_batches(
        lambda b: {"nbytes": np.array([sum(len(x) for x in b["image"])])},
        batch_size=512,
    ).sum("nbytes")
    print(f"Raw image bytes: {total_image_bytes / 1e9:.3f} GB")
else:
    # Lazy: don't count bytes here — a .sum() pass would trigger a full extra
    # generation. raw_image_bytes is recovered post-write from on-disk size instead.
    total_image_bytes = None
    print(f"Generation pipeline built (lazy — streams into the single enabled write path)")


## Write — Delta (path references)

The Databricks-native image pattern: JPEG bytes are written out as files in a Volume, and the Delta table holds an `image_path` string plus the metadata columns (no inline bytes). Writing files and the metadata table both fan out across Ray. The table is created via `ray.data.write_databricks_table` (local Spark, 2 reserved workers).

In [0]:
_images_dir = images_dir


def write_images_and_meta(batch):
    """Write each JPEG to the Volume; return the metadata row with image_path (no bytes)."""
    import os
    paths = []
    for _id, jpeg in zip(batch["id"], batch["image"]):
        p = os.path.join(_images_dir, f"{int(_id):012d}.jpg")
        with open(p, "wb") as f:
            f.write(jpeg)
        paths.append(p)
    return {
        "id":         batch["id"],
        "image_path": np.asarray(paths, dtype=object),
        "caption":    batch["caption"],
        "embedding":  batch["embedding"],
        "category":   batch["category"],
        "brightness": batch["brightness"],
        "quality":    batch["quality"],
    }


if WRITE_PATH_REF:
    t0 = time.time()
    meta_ds = ds.map_batches(write_images_and_meta, batch_size=512).materialize()
    files_write_s = time.time() - t0
    img_bytes, img_files = dir_stats(images_dir)
    # Lazy single-path run skipped the pre-write sum; the landed JPEGs ARE the raw
    # bytes (written verbatim), so recover the metric from on-disk size here.
    if total_image_bytes is None:
        total_image_bytes = img_bytes
    print(f"Delta JPEG files: {files_write_s:6.2f}s | {img_files:,} files | {img_bytes / 1e9:.3f} GB")
else:
    print("[path-ref] Skipped — set write_path_ref=true to run")


In [0]:
# Write the metadata table to Delta via local Spark (2 reserved workers).
if WRITE_PATH_REF:
    os.environ['RAY_UC_VOLUMES_FUSE_TEMP_DIR'] = ray_tmp_path

    t0 = time.time()
    meta_ds.write_databricks_table(
        f"{catalog}.{schema}.synthetic_delta_{size}",
        mode="overwrite",
    )
    delta_write_s = time.time() - t0

    delta_count = spark.sql(f"SELECT COUNT(*) AS n FROM {delta_table}").collect()[0]["n"]
    print(f"Delta table   : {delta_write_s:6.2f}s | {delta_count:,} rows written")
else:
    print("[path-ref] Skipped")


In [0]:
import pandas as pd

if WRITE_PATH_REF:
    # Delta on-disk = metadata Parquet + the referenced JPEG files.
    delta_meta_bytes = 0
    try:
        detail = spark.sql(f"DESCRIBE DETAIL {delta_table}").collect()[0]
        delta_meta_bytes = detail["sizeInBytes"] or 0
    except Exception:
        pass
    delta_total_bytes = delta_meta_bytes + img_bytes

    write_summary = pd.DataFrame([
        {"format": "delta", "write_s": round(files_write_s + delta_write_s, 2),
         "rows_per_s": round(N_ROWS / (files_write_s + delta_write_s)),
         "on_disk_GB": round(delta_total_bytes / 1e9, 3),
         "files": img_files + 1, "compression_x": round(total_image_bytes / max(1, img_bytes), 2)},
    ])
    display(write_summary)
else:
    print("[path-ref] Skipped")


## ETL benchmark — backfill a new column

Compute a derived column once and add it to the existing dataset. Lance's `add_columns` writes only the new column; Delta must `ALTER TABLE ADD COLUMN` then backfill, which rewrites the affected Parquet files. Derived column: the L2 norm of the embedding — a stand-in for any UDF-computed feature.

The **bytes rewritten** are read from the `UPDATE` operation's own `operationMetrics` (`numAddedBytes`) via `DESCRIBE HISTORY` — the true rewrite cost. The post-UPDATE table *size* is not used for this, since without a `VACUUM` it still counts the retained previous version and would overstate the write.

In [0]:
# ── Delta: ALTER TABLE ADD COLUMN + backfill (rewrites affected files) ─────
# ADD COLUMN is metadata-only (instant); skip if it exists from a previous run.
# The UPDATE below is the expensive step and is idempotent (overwrites all rows).
if WRITE_PATH_REF:
    _existing = {f.name for f in spark.table(delta_table).schema.fields}

    t0 = time.time()
    if "embedding_norm" not in _existing:
        spark.sql(f"ALTER TABLE {delta_table} ADD COLUMN embedding_norm FLOAT")
    # Embedding is stored as an array column; aggregate_norm via SQL higher-order function.
    spark.sql(f"""
        UPDATE {delta_table}
        SET embedding_norm = SQRT(AGGREGATE(TRANSFORM(embedding, x -> x * x), CAST(0.0 AS DOUBLE), (acc, v) -> acc + v))
    """)
    delta_backfill_s = time.time() - t0

    # Real bytes/files rewritten by the UPDATE — from the operation's own metrics, NOT
    # the post-UPDATE table size (which double-counts the retained old version pre-VACUUM).
    # operationMetrics values are strings; keys vary slightly by DBR — fall back gracefully.
    _hist = spark.sql(f"DESCRIBE HISTORY {delta_table}").collect()
    _upd = next(r for r in _hist if r["operation"] == "UPDATE")
    _m = _upd["operationMetrics"] or {}
    delta_bytes_rewritten = int(_m.get("numAddedBytes") or _m.get("rewrittenBytes") or 0)
    delta_files_rewritten = int(_m.get("numAddedFiles") or 0)
    delta_meta_after = spark.sql(f"DESCRIBE DETAIL {delta_table}").collect()[0]["sizeInBytes"] or 0
    print(f"Delta backfill    : {delta_backfill_s:6.2f}s | rewrote {delta_bytes_rewritten / 1e6:,.1f} MB "
          f"across {delta_files_rewritten} files (from operationMetrics)")
    print(f"  UPDATE metrics  : {dict(_m)}")
else:
    print("[path-ref] Skipped")


In [0]:
import json

# Persist Delta-side metrics for notebook 03 to compile the cross-format view.
if WRITE_PATH_REF:
    delta_total_write_s = files_write_s + delta_write_s

    # ── common block ── identical key names across 01a / 01b (and optional 01) so 03 can
    # stack the artifacts into one table with no per-format key mapping. Format-specific
    # detail is kept below in `raw`.
    common = {
        "path_label":        "delta_pathref",              # 01a = Delta path-reference
        "write_total_s":     round(delta_total_write_s, 3), # end-to-end: files + table write
        "target_write_s":    round(delta_write_s, 3),       # table write only (file-landing excluded)
        "n_output_files":    int(img_files + 1),            # JPEGs + 1 Parquet — the small-file contrast
        "on_disk_bytes":     int(delta_total_bytes),
        "etl_backfill_s":    round(delta_backfill_s, 3),
        "etl_bytes_written": int(delta_bytes_rewritten),    # real UPDATE cost (numAddedBytes)
        "roundtrip_ok":      None,                          # 01a is the source of truth — nothing to verify against
    }

    delta_metrics = {
        "size":   size,
        "n_rows": int(N_ROWS),
        "common": common,
        "raw": {                                            # format-specific detail
            "raw_image_gb":          round(total_image_bytes / 1e9, 4),
            "files_write_s":         round(files_write_s, 3),
            "delta_write_s":         round(delta_write_s, 3),
            "delta_total_write_s":   round(delta_total_write_s, 3),
            "img_files":             int(img_files),
            "img_bytes":             int(img_bytes),
            "delta_total_bytes":     int(delta_total_bytes),
            "delta_backfill_s":      round(delta_backfill_s, 3),
            "delta_bytes_rewritten": int(delta_bytes_rewritten),
            "delta_files_rewritten": int(delta_files_rewritten),
            "delta_meta_after":      int(delta_meta_after),  # NOT the rewrite cost — retained pre-VACUUM version
        },
    }
    out_path = f"{artifacts_dir}/delta_{size}.json"
    with open(out_path, "w") as f:
        json.dump(delta_metrics, f, indent=2)
    print(f"Wrote {out_path}")
    print(json.dumps(delta_metrics, indent=2))
else:
    print("[path-ref metrics] Skipped")


<cell_type>markdown</cell_type>## Optional — inline Delta (the anti-pattern, measured)

Runs when `delta_write_mode` is `inline` or `both`. It writes the generated data to a second Delta table with the JPEG bytes stored **inline** in a `binary` column — no `image_path`, no files. In `both` mode it reuses the materialized `ds`; run solo it streams generation straight in.

This is the config the [parent README](../README.md) flags as an anti-pattern: under the shuffled random-access reads a training DataLoader does, Parquet's ~128MB row groups collapse to ~1,280 rows at ~100KB/image, so a random batch scans gigabytes to read megabytes. Measuring it here puts a real number on *why* `01a`'s paved path stores `image_path` references instead.

Writes to a **separate** table `synthetic_delta_inline_{size}` (never touches the paved `01a` output) and persists `artifacts/delta_inline_{size}.json` with the same `common` keys (`path_label="delta_inline"`), so `03` can stack all three write paths.

### Where inline Delta hits a ceiling

Two limits, and the one that bites here is the *write* path, not the format:

- **Per-cell hard cap (~2.1 GB).** Parquet stores page sizes as signed 32-bit ints and the JVM caps array length near `Integer.MAX_VALUE`, so a single `binary` value above ~2.1 GB cannot be written or read. Our JPEGs are ~30–300 KB — nowhere near it. This *would* bite for inline **video** or large multi-page documents, which is exactly the kind of payload Lance's blob layout is built for.
- **Write-path memory (the real limit at scale).** `write_databricks_table` routes through Spark, which materializes row groups in executor memory. Path-ref rows are ~100-byte strings, so the image bytes never touch Spark. Inline pushes *every* image byte through the reserved Spark workers: ~150 KB/row × N is ~150 GB at 1M and ~1.5 TB at 10M — through only 2 reserved workers on this cluster. Expect heavy spill or OOM well before 10M.

At 1m/10m the write below **warns and proceeds** rather than hard-stopping — measuring how badly inline degrades (or whether it OOMs) *is* the finding. That contrast is the point: path-ref parallelises the heavy bytes as independent file writes, while inline funnels them all through Spark.

In [0]:
# ── Optional: inline Delta write — JPEG bytes in a binary column ───────────
# Writes the whole dataset, image bytes included, straight to Delta: the bytes land
# inline in the Parquet row groups — the anti-pattern under random-access reads.
# When run alongside path-ref, reuses the materialized ds (REUSE); solo, streams lazily.
if not WRITE_INLINE_DELTA:
    print("[inline Delta] Skipped — set write_inline_delta=true to run")
else:
    if size not in ("10k", "100k"):
        # Not a hard stop: at 1m/10m the inline bytes (~150KB/row) funnel through the
        # 2 reserved Spark workers (~150GB / ~1.5TB), so expect heavy spill or OOM.
        # That write-path strain IS the finding we want to measure at scale — proceed
        # and let it succeed-slowly or fail loudly. See the markdown above.
        print(f"⚠ inline Delta at size={size}: image bytes funnel through the 2 reserved "
              f"Spark workers — expect heavy spill or OOM. Running anyway (that's the test).")

    os.environ['RAY_UC_VOLUMES_FUSE_TEMP_DIR'] = ray_tmp_path
    t0 = time.time()
    ds.write_databricks_table(inline_delta_table, mode="overwrite")
    inline_write_s = time.time() - t0

    _d = spark.sql(f"DESCRIBE DETAIL {inline_delta_table}").collect()[0]
    inline_on_disk_bytes = _d["sizeInBytes"] or 0
    inline_n_files       = _d["numFiles"] or 0
    inline_count = spark.sql(f"SELECT COUNT(*) AS n FROM {inline_delta_table}").collect()[0]["n"]
    print(f"Inline Delta  : {inline_write_s:6.2f}s | {inline_count:,} rows | "
          f"{inline_on_disk_bytes / 1e9:.3f} GB across {inline_n_files} files")
    print(f"  bytes live inline in row groups here, not as {inline_count:,} separate JPEG files (01a path-ref)")


In [0]:
# Shutdown ray cluster to free up resources for Spark to do ETL backfill
try:
    shutdown_ray_cluster()
except Exception:
    pass

# ── Optional: inline Delta ETL backfill — rewrites row groups WITH image bytes ─
# Same ALTER + UPDATE as the path-ref table, but here every rewritten row group
# drags the inline JPEG bytes along — so numAddedBytes reflects the inflated
# rewrite cost inline storage imposes on any column backfill.
if not WRITE_INLINE_DELTA:
    print("[inline Delta ETL] Skipped")
else:
    _existing = {f.name for f in spark.table(inline_delta_table).schema.fields}
    t0 = time.time()
    if "embedding_norm" not in _existing:
        spark.sql(f"ALTER TABLE {inline_delta_table} ADD COLUMN embedding_norm FLOAT")
    spark.sql(f"""
        UPDATE {inline_delta_table}
        SET embedding_norm = SQRT(AGGREGATE(TRANSFORM(embedding, x -> x * x), CAST(0.0 AS DOUBLE), (acc, v) -> acc + v))
    """)
    inline_backfill_s = time.time() - t0

    _hist = spark.sql(f"DESCRIBE HISTORY {inline_delta_table}").collect()
    _upd = next(r for r in _hist if r["operation"] == "UPDATE")
    _m = _upd["operationMetrics"] or {}
    inline_bytes_rewritten = int(_m.get("numAddedBytes") or _m.get("rewrittenBytes") or 0)
    inline_files_rewritten = int(_m.get("numAddedFiles") or 0)
    print(f"Inline backfill : {inline_backfill_s:6.2f}s | rewrote {inline_bytes_rewritten / 1e6:,.1f} MB "
          f"across {inline_files_rewritten} files (image bytes dragged along vs path-ref's metadata-only rows)")

In [0]:
# ── Optional: persist inline-Delta metrics (same common keys as 01a / 01b) ──
if not WRITE_INLINE_DELTA:
    print("[inline Delta metrics] Skipped")
else:
    common = {
        "path_label":        "delta_inline",                 # inline binary column — the anti-pattern
        "write_total_s":     round(inline_write_s, 3),
        "target_write_s":    round(inline_write_s, 3),        # no file-landing step — write is the whole cost
        "n_output_files":    int(inline_n_files),             # Parquet files only (bytes are inline)
        "on_disk_bytes":     int(inline_on_disk_bytes),
        "etl_backfill_s":    round(inline_backfill_s, 3),
        "etl_bytes_written": int(inline_bytes_rewritten),     # inflated: row groups carry image bytes
        "roundtrip_ok":      None,                            # source of truth is 01a's ds — nothing to verify against
    }
    # raw_image_gb is unknown on a solo lazy inline run (no pre-write sum, no landed
    # JPEGs to measure); inline on-disk bytes already capture the stored size.
    _raw_image_gb = round(total_image_bytes / 1e9, 4) if total_image_bytes is not None else None
    inline_metrics = {
        "size":   size,
        "n_rows": int(N_ROWS),
        "common": common,
        "raw": {
            "raw_image_gb":           _raw_image_gb,
            "inline_write_s":         round(inline_write_s, 3),
            "inline_on_disk_bytes":   int(inline_on_disk_bytes),
            "inline_n_files":         int(inline_n_files),
            "inline_backfill_s":      round(inline_backfill_s, 3),
            "inline_bytes_rewritten": int(inline_bytes_rewritten),
            "inline_files_rewritten": int(inline_files_rewritten),
        },
    }
    out_path = f"{artifacts_dir}/delta_inline_{size}.json"
    with open(out_path, "w") as f:
        json.dump(inline_metrics, f, indent=2)
    print(f"Wrote {out_path}")
    print(json.dumps(inline_metrics, indent=2))


## Done — Delta artifact ready

The Delta table + referenced JPEG files are the paved Delta dataset `02_training_benchmark.ipynb` trains on. Metrics are in `artifacts/delta_{size}.json`.

**Next:** `01b_lance_native.ipynb` — write the *same* generated source straight to Lance fragments (no per-image files), then compare the two native routes in `03_compile_results.ipynb`.